### Consolidação OURO: Modelagem Dimensional

### Introdução

Neste notebook o objetivo é utilizar a tabela limpa gerada na etapa prata ('cubesat_eps.prata.eps_telemetria') e a reorganiza em um **esquema estrela**, com tabelas fato (as medições) e tabelas dimensão (o contexto das medições: qual satélite, qual sessão de coleta).

A telemetria de um cubesat tem particularidades que influenciam o modelo:
- Ciclo de órbita: o satélite alterna entre sol e eclipse a cada 90 minutos, então a tensão dos paineis oscila naturalmente. Isso não é falha.
- Falhas reais: UGUISU e RAAVANA tiveram falha documentada em painel solar. O modelo precisa permitir comparar satélites e localizar essa falha no tempo.
- Frames corrompidos: na validação prata encontramos 8 linhas com telemetria de bateria zerada, que é sinal de frame perdido e não de leitura real. A ouro é onde tratamos isso (as 3 colunas de bateria ficam nulas, e o restante da linha é preservado).

| Tabela | Tipo | Grão (1 linha =) |
|---|---|---|
| `fato_telemetria_eps` | Fato | Uma leitura de um satélite em um instante de uma sessão |
| `dim_satelite` | Dimensão | Um satélite |
| `dim_sessao` | Dimensão | Uma sessão de coleta (aba da planilha) |

#### Perguntas de negócio

1. Como a tensão/corrente dos paineis variam ao longo da órbita e da vida útil?
2. É possível identificar o momento da falha do painel em UGUISU e RAAVANA?
3. Como se compara o desempenho entre os 4 satélites?
4. Há degradação progressiva da bateria ao longo do tempo?
5. Qual face contribui mais para a geração de energia?

In [0]:
CATALOG = "cubesat_eps"
SCHEMA_PRATA = "prata"
SCHEMA_OURO = "ouro"

TABELA_PRATA = f"{CATALOG}.{SCHEMA_PRATA}.eps_telemetria"

spark.sql("DESCRIBE cubesat_eps.prata.eps_telemetria").show(50, truncate=False)

+--------------+---------+-------+
|col_name      |data_type|comment|
+--------------+---------+-------+
|time_stamp_s  |double   |NULL   |
|tpy_raw       |string   |NULL   |
|tpy_c         |double   |NULL   |
|tpx_raw       |string   |NULL   |
|tpx_c         |double   |NULL   |
|tmz_raw       |string   |NULL   |
|tmz_c         |double   |NULL   |
|tmx_raw       |string   |NULL   |
|tmx_c         |double   |NULL   |
|tpz_raw       |string   |NULL   |
|tpz_c         |double   |NULL   |
|vpy_raw       |string   |NULL   |
|vpy_mv        |double   |NULL   |
|vpx_raw       |string   |NULL   |
|vpx_mv        |double   |NULL   |
|vmz_raw       |string   |NULL   |
|vmz_mv        |double   |NULL   |
|vmx_raw       |string   |NULL   |
|vmx_mv        |double   |NULL   |
|vpz_raw       |string   |NULL   |
|vpz_mv        |double   |NULL   |
|ipy_raw       |string   |NULL   |
|ipy_ma        |double   |NULL   |
|ipx_raw       |string   |NULL   |
|ipx_ma        |double   |NULL   |
|imz_raw       |stri

### Dimensão 'dim_satelite'

Esta dimensão descreve o contexto fixo de cada satélite: qual constelação ele pertence e se teve falha de painel solar documentada na literatura (UGUISU e RAAVANA). Esses atributos não vem da telemetria em si, mas são conhecimento de domínio sobre a missão - por isso vamos declará-los manualmente, mas primeiro conferimos os valores exatos de 'satellite_id' que existem na prata, para garantir que o texto bate exatamente quando formos montar a tabela.

In [0]:
spark.sql("SELECT DISTINCT satellite_id FROM cubesat_eps.prata.eps_telemetria ORDER BY satellite_id").show()

+------------+
|satellite_id|
+------------+
|   NEPALISAT|
|     RAAVANA|
|       TSURU|
|      UGUISU|
+------------+



#### Montando a dimensão 'dim_satelite'

Esta dimensão traz o contexto fixo de cada satélite. Os dois atributos que valem a pena guardar aqui são:

- **constelação**:  todos são da constelação BIRDS - parece redundante hoje (só temos uma constelação), mas documenta explicitamente esse fato e deixa o modelo pronto caso o dataset seja estendido com outra constelação no futuro.
- **falha_painel_documentada**: flag booleana vinda da literatura do dataset (RAAVANA e UGUISU tiveram falha de painel solar documentada). É conhecimento de domínio, não algo calculável a partir da telemetria.

In [0]:
from pyspark.sql import Row

dim_satelite_data = [
    Row(satellite_id="NEPALISAT", constelacao="BIRDS", falha_painel_documentada=False),
    Row(satellite_id="RAAVANA",   constelacao="BIRDS", falha_painel_documentada=True),
    Row(satellite_id="TSURU",     constelacao="BIRDS", falha_painel_documentada=False),
    Row(satellite_id="UGUISU",    constelacao="BIRDS", falha_painel_documentada=True),
]

dim_satelite = spark.createDataFrame(dim_satelite_data)
dim_satelite.show()

+------------+-----------+------------------------+
|satellite_id|constelacao|falha_painel_documentada|
+------------+-----------+------------------------+
|   NEPALISAT|      BIRDS|                   false|
|     RAAVANA|      BIRDS|                    true|
|       TSURU|      BIRDS|                   false|
|      UGUISU|      BIRDS|                    true|
+------------+-----------+------------------------+



#### Gravando 'dim_satelite'

Como é a dimensão pequena e estática (montada manualmente, não derivada da telemetria), o 'overwrite' aqui é seguro e correto: cada execução do notebook recria a tabela do zero a partir dessa lista fixa, sem risco de duplicar ou acumular lixo de execuções anteriores.

In [0]:
dim_satelite.write.mode("overwrite").saveAsTable(f"{CATALOG}.ouro.dim_satelite")

spark.sql(f"SELECT * FROM {CATALOG}.ouro.dim_satelite").show()

+------------+-----------+------------------------+
|satellite_id|constelacao|falha_painel_documentada|
+------------+-----------+------------------------+
|   NEPALISAT|      BIRDS|                   false|
|     RAAVANA|      BIRDS|                    true|
|       TSURU|      BIRDS|                   false|
|      UGUISU|      BIRDS|                    true|
+------------+-----------+------------------------+



### Dimensão 'dim_sessao'

Cada sessão de coleta corresponde a uma aba de planilha original ('sheet_name'), associada a um satélite ('satellite_id'), com uma data ('sheet_date', que pode ser nula para testes em solo) e uma flag indicando se foi teste em solo ou operação em órbita ('is_ground_test').

Antes de decidir como gerar a chave substituta ('sessao_id') dessa dimensão, precisamos entender a granularidade real: será que 'sheet_name' já é único por satélite, ou pode se repetir entre satélites diferentes (ex: duas planilhas usando o nome de aba "Sheet1")? Isso decide se a chave natural precisa ser '(satellite_id, sheet_name)' ou só 'sheet_name'.

In [0]:
spark.sql("""
    SELECT satellite_id, sheet_name, sheet_date, is_ground_test, source_file, COUNT(*) AS n_leituras
    FROM cubesat_eps.prata.eps_telemetria
    GROUP BY satellite_id, sheet_name, sheet_date, is_ground_test, source_file
    ORDER BY satellite_id, sheet_date
""").show(100, truncate=False)

+------------+---------------+----------+--------------+--------------+----------+
|satellite_id|sheet_name     |sheet_date|is_ground_test|source_file   |n_leituras|
+------------+---------------+----------+--------------+--------------+----------+
|NEPALISAT   |9 Nov 2020     |2020-11-09|false         |NEPALISAT.xlsx|1080      |
|NEPALISAT   |25 Nov 2020    |2020-11-25|false         |NEPALISAT.xlsx|880       |
|NEPALISAT   |10 April 2021  |2021-04-10|false         |NEPALISAT.xlsx|1080      |
|RAAVANA     |13 Feb 2021    |2021-02-13|false         |RAAVANA.xlsx  |1079      |
|RAAVANA     |11 March 2021  |2021-03-11|false         |RAAVANA.xlsx  |1080      |
|TSURU       |Test1 w batt   |NULL      |true          |TSURU.xlsx    |556       |
|TSURU       |Test2 wo batt  |NULL      |true          |TSURU.xlsx    |556       |
|TSURU       |March 28       |2021-03-28|false         |TSURU.xlsx    |1095      |
|TSURU       |March 31       |2021-03-31|false         |TSURU.xlsx    |359       |
|TSU

#### Montando 'dim_sessao'

Para gerar 'sessao_id', usamos 'row_number()' sobre uma janela ordenada por 'satellite_id', 'sheet_date', 'sheet_name'. A ordenação é determinística (mesmos critérios sempre geram a mesma sequência), o que é importante aqui: mesmo recriando tabela do zero a cada execução do notebook ('overwrite'), os IDs de sessão vão sair sempre iguais, e não vão "embaralhar" a cada rerun.

Observação: usando "row_number()" sem "partitionBy" faz o Spark mover todos os dados para uma única partição para numerar em ordem global. Para uma dimensão pequena como essa (aproximadamente 50 linhas), isso é totalmente aceitável, não seria uma boa prática numa tabela de milhões de linhas, mas aqui o custo é irrelevante.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sessoes_distintas = spark.sql(f"""
    SELECT DISTINCT satellite_id, sheet_name, sheet_date, is_ground_test, source_file
    FROM {TABELA_PRATA}
""")

janela_ordenacao = Window.orderBy("satellite_id", "sheet_date", "sheet_name")

dim_sessao = sessoes_distintas.withColumn(
    "sessao_id", F.row_number().over(janela_ordenacao)
).select("sessao_id", "satellite_id", "sheet_name", "sheet_date", "is_ground_test", "source_file")

dim_sessao.orderBy("sessao_id").show(60, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+------------+---------------+----------+--------------+--------------+
|sessao_id|satellite_id|sheet_name     |sheet_date|is_ground_test|source_file   |
+---------+------------+---------------+----------+--------------+--------------+
|1        |NEPALISAT   |9 Nov 2020     |2020-11-09|false         |NEPALISAT.xlsx|
|2        |NEPALISAT   |25 Nov 2020    |2020-11-25|false         |NEPALISAT.xlsx|
|3        |NEPALISAT   |10 April 2021  |2021-04-10|false         |NEPALISAT.xlsx|
|4        |RAAVANA     |13 Feb 2021    |2021-02-13|false         |RAAVANA.xlsx  |
|5        |RAAVANA     |11 March 2021  |2021-03-11|false         |RAAVANA.xlsx  |
|6        |TSURU       |Test1 w batt   |NULL      |true          |TSURU.xlsx    |
|7        |TSURU       |Test2 wo batt  |NULL      |true          |TSURU.xlsx    |
|8        |TSURU       |March 28       |2021-03-28|false         |TSURU.xlsx    |
|9        |TSURU       |March 31       |2021-03-31|false         |TSURU.xlsx    |
|10       |TSURU

#### Gravando 'dim_sessao'

Mesma lógica da `dim_satelite`: `overwrite`, porque a dimensão é inteiramente recalculada a partir da prata a cada execução.

In [0]:
dim_sessao.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA_OURO}.dim_sessao")

spark.sql(f"SELECT COUNT(*) AS total_sessoes FROM {CATALOG}.{SCHEMA_OURO}.dim_sessao").show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------+
|total_sessoes|
+-------------+
|           50|
+-------------+



### Identificando frames de bateria corrompidos

Na validação prata identificamos registros onde a leitura de bateria ('vbat_v', 'ibat_ma', 'tbat_c') não reflete uma leitura física real, sinal de um frame de telemetria perdido/corrompido, e não uma medição válida. O padrão característico é 'ibat_ma = 0' e 'tbat_c = 0' simultaneamente: é fisicamente inverossímel a corrente da bateria e a temperatura da bateria zerarem exatamente ao mesmo tempo durante uma operação normal (mesmo em eclipse a bateria está sempre drenando ou carregando alguma corrente, e a temperatura não é um sensor digital que reporte exatamente 0.0).

In [0]:
frames_corrompidos = spark.sql(f"""
    SELECT satellite_id, sheet_name, time_stamp_s, vbat_v, ibat_ma, tbat_c
    FROM {TABELA_PRATA}
    WHERE ibat_ma = 0 AND tbat_c = 0
    ORDER BY satellite_id, sheet_name, time_stamp_s
""")

frames_corrompidos.show(20, truncate=False)
print(f"Total de frames corrompidos: {frames_corrompidos.count()}")

+------------+-----------+------------+------+-------+------+
|satellite_id|sheet_name |time_stamp_s|vbat_v|ibat_ma|tbat_c|
+------------+-----------+------------+------+-------+------+
|TSURU       |August 26  |5890.0      |0.0   |0.0    |0.0   |
|TSURU       |August 26  |5900.0      |0.0   |0.0    |0.0   |
|TSURU       |August 26  |5910.0      |0.0   |0.0    |0.0   |
|TSURU       |August 26  |5920.0      |0.0   |0.0    |0.0   |
|TSURU       |August 26  |5930.0      |0.0   |0.0    |0.0   |
|TSURU       |August 26  |5940.0      |0.0   |0.0    |0.0   |
|TSURU       |March 31   |4045.4      |0.0   |0.0    |0.0   |
|UGUISU      |13 Feb 2021|5355.0      |6.44  |0.0    |0.0   |
+------------+-----------+------------+------+-------+------+

Total de frames corrompidos: 8


### Juntando a prata com 'dim_sessao' para obter 'sessao_id'

Fazemos o join pela chave natural '(satellite_id, sheet_name)' que já validamos ser única em 'dim_sessao'. Antes de seguir para a seleção final das colunas, conferimos que a contagem das linhas do resultado do join é idêntica à contagem original prata (32837), caso apareça diferente é sinal de join incorreto. 

In [0]:
prata_df = spark.table(TABELA_PRATA)
dim_sessao_ref = spark.table(f"{CATALOG}.{SCHEMA_OURO}.dim_sessao").select("sessao_id", "satellite_id", "sheet_name")

fato_base = prata_df.join(dim_sessao_ref, on=["satellite_id", "sheet_name"], how="inner")

print(f"Linhas na prata original: {prata_df.count()}")
print(f"Linhas após o join com dim_sessao: {fato_base.count()}")

Linhas na prata original: 32837
Linhas após o join com dim_sessao: 32837


### Montando 'fato_telemetria_eps'

Grão: é uma leitura de um satélite em um instante ('time_stamp_s') de uma sessão ('sessao_id').
Colunas mantidas: o timestamp da leitura, as 5 temperaturas de painel ('t*_c'), as 5 tensões de painel ('v*_mv'), as 5 correntes de painel ('i*_ma') e as 3 leituras da mesma linha (paineis e temperaturas continuam válidos nesses casos).

As colunas '_raw' da prata não entram nesta etapa: elas existiam para rastrear o valor bruto antes da calibração, o que é relevante para auditoria de qualidade de dados, mas não agrega valor analítico nesta camada de consumo.

In [0]:
from pyspark.sql import functions as F

condicao_frame_corrompido = (F.col("ibat_ma") == 0) & (F.col("tbat_c") == 0)

fato_telemetria_eps = fato_base.select(
    "sessao_id",
    "time_stamp_s",
    "tpy_c", "tpx_c", "tmz_c", "tmx_c", "tpz_c",
    "vpy_mv", "vpx_mv", "vmz_mv", "vmx_mv", "vpz_mv",
    "ipy_ma", "ipx_ma", "imz_ma", "imx_ma", "ipz_ma",
    F.when(condicao_frame_corrompido, None).otherwise(F.col("vbat_v")).alias("vbat_v"),
    F.when(condicao_frame_corrompido, None).otherwise(F.col("ibat_ma")).alias("ibat_ma"),
    F.when(condicao_frame_corrompido, None).otherwise(F.col("tbat_c")).alias("tbat_c"),
)

print(f"Total de linhas: {fato_telemetria_eps.count()}")
print(f"Linhas com vbat_v nulo (esperado: 8): {fato_telemetria_eps.filter(F.col('vbat_v').isNull()).count()}")

Total de linhas: 32837
Linhas com vbat_v nulo (esperado: 8): 8


### Gravando 'fato_telemetria_eps' na camada ouro

Assim como as dimensões, usamos 'overwrite': a fato inteira é recalculada a partir da prata a cada execução do notebook, então não há risco de duplicação.

In [0]:
fato_telemetria_eps.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA_OURO}.fato_telemetria_eps")

spark.sql(f"""
    SELECT COUNT(*) AS total_linhas,
           COUNT(vbat_v) AS linhas_com_bateria_valida
    FROM {CATALOG}.{SCHEMA_OURO}.fato_telemetria_eps
""").show()

+------------+-------------------------+
|total_linhas|linhas_com_bateria_valida|
+------------+-------------------------+
|       32837|                    32829|
+------------+-------------------------+



### Checagem de integridade referencial

Vamos confirmar que não existem 'chaves_órfãs': nenhuma linha de fato aponta para um 'sessao_id' inexistente em 'dim_sessao', e nenhuma sessão aponta para um 'satellite_id' inexistente em 'dim_satellite'. Isso é esperado já que construímos a fato a partir de um inner join com a própria 'dim_sessao', mas é uma boa prática documentar essa validação explicitamente no notebook.

In [0]:
orfas_fato_sessao = spark.sql(f"""
    SELECT COUNT(*) AS linhas_orfas
    FROM {CATALOG}.{SCHEMA_OURO}.fato_telemetria_eps f
    LEFT ANTI JOIN {CATALOG}.{SCHEMA_OURO}.dim_sessao s ON f.sessao_id = s.sessao_id
""")

orfas_sessao_satelite = spark.sql(f"""
    SELECT COUNT(*) AS sessoes_orfas
    FROM {CATALOG}.{SCHEMA_OURO}.dim_sessao s
    LEFT ANTI JOIN {CATALOG}.{SCHEMA_OURO}.dim_satelite d ON s.satellite_id = d.satellite_id
""")

orfas_fato_sessao.show()
orfas_sessao_satelite.show()

+------------+
|linhas_orfas|
+------------+
|           0|
+------------+

+-------------+
|sessoes_orfas|
+-------------+
|            0|
+-------------+



### Catálogo de Dados: 'dim_satellite'

Vamos usar os comentários nativos do Unity Catalog ('COMMENT ON TABLE' e 'ALTER TABLE ... ALTER COLUMN ... COMMENT') em vez de escrever a documentação só em markdown solto. Isso tem duas vantagens: aparece direto no Catalog Explorer da Databricks e pode ser consultado por SQL ('information_schema.columns' ou 'DESCRIBE TABLE EXTENDED').

In [0]:
spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.{SCHEMA_OURO}.dim_satelite IS
    'Dimensão que descreve o contexto fixo de cada satélite da constelação BIRDS. Grão: um satélite por linha. Fonte: construída manualmente com conhecimento de domínio do projeto (não derivada de telemetria).'
""")

spark.sql(f"""
    ALTER TABLE {CATALOG}.{SCHEMA_OURO}.dim_satelite ALTER COLUMN satellite_id COMMENT
    'Identificador do satélite (chave primária da dimensão). Tipo: texto (string). Domínio: NEPALISAT, RAAVANA, TSURU, UGUISU. Linhagem: extraído do nome do arquivo Excel de origem na camada bronze (um arquivo por satélite).'
""")

spark.sql(f"""
    ALTER TABLE {CATALOG}.{SCHEMA_OURO}.dim_satelite ALTER COLUMN constelacao COMMENT
    'Constelação de cubesats à qual o satélite pertence. Tipo: texto (string). Domínio: valor único "BIRDS" neste dataset. Linhagem: atribuído manualmente com base na documentação do dataset original (Mendeley Data), não presente nos arquivos brutos.'
""")

spark.sql(f"""
    ALTER TABLE {CATALOG}.{SCHEMA_OURO}.dim_satelite ALTER COLUMN falha_painel_documentada COMMENT
    'Indica se há registro documentado (na literatura da missão) de falha de painel solar nesse satélite. Tipo: booleano. Domínio: true (RAAVANA, UGUISU) ou false (NEPALISAT, TSURU). Linhagem: atribuído manualmente com base em documentação externa da missão BIRDS, não calculado a partir da telemetria.'
""")

spark.sql(f"DESCRIBE TABLE EXTENDED {CATALOG}.{SCHEMA_OURO}.dim_satelite").show(50, truncate=False)

+----------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|col_name                    |data_type                                                                                                                                                                                                                                                              

### Catálogo de Dados: 'dim_sessao'


In [0]:
spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.{SCHEMA_OURO}.dim_sessao IS
    'Dimensão que descreve cada sessão de coleta de telemetria (uma aba da planilha original de um satélite). Grão: uma sessão por linha. Fonte: derivada da camada prata, com chave substituta gerada na modelagem da ouro.'
""")

spark.sql(f"""
    ALTER TABLE {CATALOG}.{SCHEMA_OURO}.dim_sessao ALTER COLUMN sessao_id COMMENT
    'Chave substituta (surrogate key) da sessão, chave primária da dimensão. Tipo: número inteiro. Domínio: 1 a 50 (sem furos). Linhagem: gerado via row_number() sobre janela ordenada por (satellite_id, sheet_date, sheet_name) durante a modelagem da ouro; não existe na fonte original.'
""")

spark.sql(f"""
    ALTER TABLE {CATALOG}.{SCHEMA_OURO}.dim_sessao ALTER COLUMN satellite_id COMMENT
    'Identificador do satélite ao qual a sessão pertence (chave estrangeira para dim_satelite). Tipo: texto (string). Domínio: NEPALISAT, RAAVANA, TSURU, UGUISU. Linhagem: extraído do nome do arquivo Excel de origem na camada bronze.'
""")

spark.sql(f"""
    ALTER TABLE {CATALOG}.{SCHEMA_OURO}.dim_sessao ALTER COLUMN sheet_name COMMENT
    'Nome da aba da planilha original que originou a sessão. Tipo: texto (string). Domínio: datas em formatos variados (ex: "13 Feb 2021", "March 28") ou rótulos de teste em solo (ex: "Test1 w batt", "Test2 wo batt"). Linhagem: nome literal da aba do arquivo Excel, capturado na leitura da camada bronze.'
""")

spark.sql(f"""
    ALTER TABLE {CATALOG}.{SCHEMA_OURO}.dim_sessao ALTER COLUMN sheet_date COMMENT
    'Data da sessão de coleta, quando aplicável. Tipo: data. Domínio: 2020-10-23 a 2022-01-24, ou nulo para sessões de teste em solo (2 sessões nulas, ambas do TSURU). Linhagem: parseada do texto de sheet_name por uma função de parsing (parse_sheet_date) na etapa bronze->prata.'
""")

spark.sql(f"""
    ALTER TABLE {CATALOG}.{SCHEMA_OURO}.dim_sessao ALTER COLUMN is_ground_test COMMENT
    'Indica se a sessão foi um teste em solo (bancada) em vez de operação em órbita. Tipo: booleano. Domínio: true (2 sessões do TSURU) ou false (48 sessões restantes). Linhagem: derivado do nome da aba (sheet_name) na etapa bronze->prata.'
""")

spark.sql(f"""
    ALTER TABLE {CATALOG}.{SCHEMA_OURO}.dim_sessao ALTER COLUMN source_file COMMENT
    'Nome do arquivo Excel de origem da sessão. Tipo: texto (string). Domínio: NEPALISAT.xlsx, RAAVANA.xlsx, TSURU.xlsx, UGUISU.xlsx. Linhagem: nome do arquivo lido do volume do Unity Catalog na etapa de ingestão bronze.'
""")

spark.sql(f"DESCRIBE TABLE EXTENDED {CATALOG}.{SCHEMA_OURO}.dim_sessao").show(50, truncate=False)

+----------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|col_name                    |data_type                                                                                                                                                                                                                                                            

### Levantando o domínio de valores de 'fato_telemetria_eps'

Antes de documentar o catálogo, calculamos min/max reais de cada coluna numérica da fato.

In [0]:
from pyspark.sql import functions as F

fato_df = spark.table(f"{CATALOG}.{SCHEMA_OURO}.fato_telemetria_eps")

colunas_numericas = [
    "time_stamp_s",
    "tpy_c", "tpx_c", "tmz_c", "tmx_c", "tpz_c",
    "vpy_mv", "vpx_mv", "vmz_mv", "vmx_mv", "vpz_mv",
    "ipy_ma", "ipx_ma", "imz_ma", "imx_ma", "ipz_ma",
    "vbat_v", "ibat_ma", "tbat_c",
]

agregacoes = []
for coluna in colunas_numericas:
    agregacoes.append(F.min(coluna).alias(f"min_{coluna}"))
    agregacoes.append(F.max(coluna).alias(f"max_{coluna}"))

fato_df.agg(*agregacoes).show(n=1, vertical=True, truncate=False)

-RECORD 0-------------------
 min_time_stamp_s | 0.0     
 max_time_stamp_s | 12362.2 
 min_tpy_c        | -28.33  
 max_tpy_c        | 87.75   
 min_tpx_c        | -30.21  
 max_tpx_c        | 81.2    
 min_tmz_c        | -32.32  
 max_tmz_c        | 89.64   
 min_tmx_c        | -28.44  
 max_tmx_c        | 91.97   
 min_tpz_c        | -32.21  
 max_tpz_c        | 69.0    
 min_vpy_mv       | 0.0     
 max_vpy_mv       | 5633.39 
 min_vpx_mv       | 0.0     
 max_vpx_mv       | 5627.29 
 min_vmz_mv       | 0.0     
 max_vmz_mv       | 5711.23 
 min_vmx_mv       | 0.0     
 max_vmx_mv       | 5630.34 
 min_vpz_mv       | 0.0     
 max_vpz_mv       | 5572.34 
 min_ipy_ma       | 0.0     
 max_ipy_ma       | 540.69  
 min_ipx_ma       | 0.0     
 max_ipx_ma       | 536.03  
 min_imz_ma       | 0.0     
 max_imz_ma       | 489.42  
 min_imx_ma       | 0.0     
 max_imx_ma       | 508.06  
 min_ipz_ma       | 0.0     
 max_ipz_ma       | 484.75  
 min_vbat_v       | 3.79    
 max_vbat_v   

### Catálogo de Dados: 'fato_telemetria_eps'

Aqui documentamos a tabela fato com os domínios reais calculados na célula anterior. Note que 'vbat_v', 'ibat_ma' e 'tbat_c' têm a observação explícita sobre os 8 valores nulos intencionais (frames corrompidos), isso é exatamente o tipo de decisão de qualidade de dados que o catálogo precisa registrar, para quem consumir a tabela não interpretar esses nulos como erro de pipeline.

In [0]:
comentarios_fato = {
    "sessao_id": "Chave estrangeira para dim_sessao.sessao_id, identifica a sessão de coleta à qual a leitura pertence. Tipo: número inteiro. Domínio: 1 a 50. Linhagem: obtido via join da prata com dim_sessao pela chave natural (satellite_id, sheet_name) na modelagem da ouro.",
    "time_stamp_s": "Instante da leitura, em segundos desde o início da sessão de coleta. Tipo: decimal (double). Domínio: 0.0 a 12362.2 (~3h26min). Linhagem: coluna original da prata, sem transformação adicional na ouro.",
    "tpy_c": "Temperatura do painel na face +Y. Tipo: decimal (double), em graus Celsius. Domínio: -28.33 a 87.75. Linhagem: calibrado a partir do valor bruto do sensor (tpy_raw) na etapa bronze->prata; a oscilação ampla reflete o ciclo órbita-eclipse do satélite, não é anomalia.",
    "tpx_c": "Temperatura do painel na face +X. Tipo: decimal (double), em graus Celsius. Domínio: -30.21 a 81.2. Linhagem: calibrado a partir do valor bruto do sensor (tpx_raw) na etapa bronze->prata.",
    "tmz_c": "Temperatura do painel na face -Z. Tipo: decimal (double), em graus Celsius. Domínio: -32.32 a 89.64. Linhagem: calibrado a partir do valor bruto do sensor (tmz_raw) na etapa bronze->prata.",
    "tmx_c": "Temperatura do painel na face -X. Tipo: decimal (double), em graus Celsius. Domínio: -28.44 a 91.97. Linhagem: calibrado a partir do valor bruto do sensor (tmx_raw) na etapa bronze->prata.",
    "tpz_c": "Temperatura do painel na face +Z. Tipo: decimal (double), em graus Celsius. Domínio: -32.21 a 69.0. Linhagem: calibrado a partir do valor bruto do sensor (tpz_raw) na etapa bronze->prata.",
    "vpy_mv": "Tensão gerada pelo painel na face +Y. Tipo: decimal (double), em milivolts. Domínio: 0.0 a 5633.39. Linhagem: calibrado a partir do valor bruto do sensor (vpy_raw) na etapa bronze->prata. Em eclipse orbital a tensão fica em torno de 1.200-1.300 mV (piso do sensor/albedo), não em zero; valores próximos de 0 ocorrem na RAAVANA, cuja face +Y está inoperante.",
    "vpx_mv": "Tensão gerada pelo painel na face +X. Tipo: decimal (double), em milivolts. Domínio: 0.0 a 5627.29. Linhagem: calibrado a partir do valor bruto do sensor (vpx_raw) na etapa bronze->prata.",
    "vmz_mv": "Tensão gerada pelo painel na face -Z. Tipo: decimal (double), em milivolts. Domínio: 0.0 a 5711.23. Linhagem: calibrado a partir do valor bruto do sensor (vmz_raw) na etapa bronze->prata.",
    "vmx_mv": "Tensão gerada pelo painel na face -X. Tipo: decimal (double), em milivolts. Domínio: 0.0 a 5630.34. Linhagem: calibrado a partir do valor bruto do sensor (vmx_raw) na etapa bronze->prata.",
    "vpz_mv": "Tensão gerada pelo painel na face +Z. Tipo: decimal (double), em milivolts. Domínio: 0.0 a 5572.34. Linhagem: calibrado a partir do valor bruto do sensor (vpz_raw) na etapa bronze->prata.",
    "ipy_ma": "Corrente gerada pelo painel na face +Y. Tipo: decimal (double), em miliampères. Domínio: 0.0 a 540.69. Linhagem: calibrado a partir do valor bruto do sensor (ipy_raw) na etapa bronze->prata.",
    "ipx_ma": "Corrente gerada pelo painel na face +X. Tipo: decimal (double), em miliampères. Domínio: 0.0 a 536.03. Linhagem: calibrado a partir do valor bruto do sensor (ipx_raw) na etapa bronze->prata.",
    "imz_ma": "Corrente gerada pelo painel na face -Z. Tipo: decimal (double), em miliampères. Domínio: 0.0 a 489.42. Linhagem: calibrado a partir do valor bruto do sensor (imz_raw) na etapa bronze->prata.",
    "imx_ma": "Corrente gerada pelo painel na face -X. Tipo: decimal (double), em miliampères. Domínio: 0.0 a 508.06. Linhagem: calibrado a partir do valor bruto do sensor (imx_raw) na etapa bronze->prata.",
    "ipz_ma": "Corrente gerada pelo painel na face +Z. Tipo: decimal (double), em miliampères. Domínio: 0.0 a 484.75. Linhagem: calibrado a partir do valor bruto do sensor (ipz_raw) na etapa bronze->prata.",
    "vbat_v": "Tensão da bateria. Tipo: decimal (double), em volts. Domínio: 3.79 a 4.22 (faixa típica de célula única de Li-ion/Li-po), com 8 valores nulos. Linhagem: calibrado a partir do valor bruto do sensor (vbat_raw) na etapa bronze->prata; nulificado na modelagem da ouro nos 8 frames de telemetria identificados como corrompidos (ibat_ma=0 e tbat_c=0 simultaneamente), preservando as demais leituras da mesma linha.",
    "ibat_ma": "Corrente da bateria (positiva = carregando, negativa = descarregando). Tipo: decimal (double), em miliampères. Domínio: -653.24 a 1200.42, com 8 valores nulos. Linhagem: calibrado a partir do valor bruto do sensor (ibat_raw) na etapa bronze->prata; nulificado na modelagem da ouro nos 8 frames de telemetria corrompidos identificados na validação de qualidade da prata.",
    "tbat_c": "Temperatura da bateria. Tipo: decimal (double), em graus Celsius. Domínio: -22.68 a 24.44, com 8 valores nulos. O mínimo de -22.68 é uma leitura espúria isolada (UGUISU, 23/10/2020, instante 0) não capturada pelo critério de limpeza; desconsiderando-a, a faixa real é de aproximadamente -0.1 a 24.44. Linhagem: calibrado a partir do valor bruto do sensor (tbat_raw) na etapa bronze->prata; nulificado na modelagem da ouro nos 8 frames de telemetria corrompidos identificados na validação de qualidade da prata.",
}

spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.{SCHEMA_OURO}.fato_telemetria_eps IS
    'Tabela fato com as leituras de telemetria do subsistema de potência elétrica (EPS) dos satélites da constelação BIRDS. Grão: uma leitura de um satélite em um instante (time_stamp_s) de uma sessão de coleta (sessao_id). Fonte: consolidada e calibrada a partir da camada prata, com as colunas de bateria nulificadas nos 8 frames identificados como corrompidos na validação de qualidade de dados.'
""")

for coluna, descricao in comentarios_fato.items():
    descricao_escapada = descricao.replace("'", "''")
    spark.sql(f"ALTER TABLE {CATALOG}.{SCHEMA_OURO}.fato_telemetria_eps ALTER COLUMN {coluna} COMMENT '{descricao_escapada}'")

print("Comentários aplicados.")

spark.sql(f"DESCRIBE TABLE EXTENDED {CATALOG}.{SCHEMA_OURO}.fato_telemetria_eps").show(60, truncate=False)

Comentários aplicados.
+----------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|col_name                    |data_type                                                                                                        